# D01 prompt ablation — run experiments + figures

Notebook driver for **`research/eval/runner.py`**: separate **code** cells invoke each **`prompt_condition`**, mirroring [`run_prompt_ablation.sh`](./run_prompt_ablation.sh).

**Downstream:** stable PNGs under **`results/figures/`** power the **markdown image** sections at the end (re-run the plotting cell after new JSONLs land).

**Phone:** set **`PHONE_IP`** or **`PHONE_URL`** in the environment before executing run cells.


## Prerequisites

**Repo & Python:** Civics checkout with `research/artifacts/`. Install from repo root: `pip install -r research/prompt_ablation/requirements-notebook.txt` and `pip install -r research/eval/requirements.txt` (`requests`, `python-Levenshtein`, scoring).

### Setting up the iPhone (eval server)

1. **Eval build:** compile the app with **`EVAL_MODE=true`** (starts HTTP **8080**). Typical: `mobile/scripts/dev_deploy.sh --eval` — details in [`research/eval/README.md`](../eval/README.md).
2. **Deploy to device:** Xcode **Product → Run** onto the physical iPhone.
3. **Verify:** device logs must show **`Eval server running on port 8080`** before any runner cell.
4. **Same Wi‑Fi** as Mac; use the phone’s **LAN IP** (Settings → Wi‑Fi → ⓘ) as **`PHONE_IP`** (runner uses **`http://<PHONE_IP>:8080`**).
5. **Quick check:** on Mac — `curl "http://192.168.x.x:8080/health"` → expect **`"status":"ok"`** in the JSON body.
6. **Thermal / stability:** plug in during long batches; **`CONDITION_BREAK_S`** (env or notebook) spaces heavy sessions.

```bash
export PHONE_IP=192.168.x.x
# optional: export CONDITION_BREAK_S=180
```

Set **`RUN_EXPERIMENTS = False`** in the next cell to **skip** phone calls and only rebuild figures from existing JSONLs.

**Jupyter:** exporting in a terminal **before** starting Jupyter works. Inside the notebook, **`!export PHONE_IP=…`** does **not** set variables for Python — use the **`_phone_ip`** line in the config cell, or **`%env PHONE_IP=192.168.x.x`**, or **`os.environ["PHONE_IP"] = "…"`** in code.


In [8]:
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path

RUN_EXPERIMENTS = True
RUNS = 20
VARIANTS = "clean,clean_jpeg,degraded,blurry"
# Align STAMP across generic + semantic + preview JSONLs; match existing filenames for plots-only runs.
STAMP = "20260412_notebook"

# Phone for runner + health check: MUST live in the Python process environment.
# Shell magics like `!export PHONE_IP=...` run a *subprocess* — they do NOT update
# `os.environ` for later cells. Options: set `_phone_ip` below, use IPython `%env PHONE_IP=...`,
# or start Jupyter from a terminal that already exported PHONE_IP / PHONE_URL.
_phone_ip = "192.168.86.241"  # e.g. "192.168.86.241"
if _phone_ip.strip():
    os.environ["PHONE_IP"] = _phone_ip.strip()

CONDITION_BREAK_S = float(os.environ.get("CONDITION_BREAK_S", "180"))



def _repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "research" / "eval" / "runner.py").is_file():
            return p
    raise RuntimeError("Cannot find research/eval/runner.py — open from civics checkout.")


NOTEBOOK_PATH = Path.cwd().resolve()
REPO_ROOT = _repo_root(NOTEBOOK_PATH)
EVAL_DIR = REPO_ROOT / "research" / "eval"
LAB_DIR = (
    NOTEBOOK_PATH
    if (NOTEBOOK_PATH / "run_prompt_ablation.sh").is_file()
    else REPO_ROOT / "research" / "prompt_ablation"
)
RESULTS_DIR = LAB_DIR / "results"
FIG_DIR = RESULTS_DIR / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

PATH_GENERIC = RESULTS_DIR / f"ablation_generic_{STAMP}.jsonl"
PATH_SEMANTIC = RESULTS_DIR / f"ablation_semantic_{STAMP}.jsonl"
PATH_PREVIEW = RESULTS_DIR / f"ablation_semantic_preview_{STAMP}.jsonl"

OUT_BY_CONDITION = {
    "generic": PATH_GENERIC,
    "semantic": PATH_SEMANTIC,
    "semantic-preview": PATH_PREVIEW,
}


def condition_break() -> None:
    if CONDITION_BREAK_S <= 0:
        return
    print(f"Cooling down {CONDITION_BREAK_S:.0f}s before next scenario…")
    time.sleep(CONDITION_BREAK_S)


def run_scenario(prompt_condition: str) -> Path:
    # Invoke runner.py for one D01 ablation condition; return output JSONL path.
    out = OUT_BY_CONDITION[prompt_condition]
    if not RUN_EXPERIMENTS:
        print(f"RUN_EXPERIMENTS=False — skip {prompt_condition!r}")
        return out
    cmd = [
        sys.executable,
        str(EVAL_DIR / "runner.py"),
        "--artifacts",
        "D01",
        "--track",
        "a",
        "--variants",
        VARIANTS,
        "--prompt-condition",
        prompt_condition,
        "--runs",
        str(RUNS),
        "--temp",
        "0.0",
        "--cooldown",
        "2.0",
        "--condition-break-s",
        "0",
        "--out",
        str(out),
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=str(EVAL_DIR), check=True, env=os.environ.copy())
    print(f"Wrote {out}")
    return out


print("EVAL_DIR:", EVAL_DIR)
print("PHONE_IP:", os.environ.get("PHONE_IP") or "(unset — set _phone_ip above or %env)")
print("PHONE_URL:", os.environ.get("PHONE_URL") or "(unset)")
print("JSONL outs:", PATH_GENERIC.name, PATH_SEMANTIC.name, PATH_PREVIEW.name)


EVAL_DIR: /Users/danfinkel/github/civics/research/eval
PHONE_IP: 192.168.86.241
PHONE_URL: (unset)
JSONL outs: ablation_generic_20260412_notebook.jsonl ablation_semantic_20260412_notebook.jsonl ablation_semantic_preview_20260412_notebook.jsonl


## Sanity check (optional)

Run the next cell once **`PHONE_IP`** (or **`PHONE_URL`**) is on the **`os.environ`** of this kernel:

- Set **`_phone_ip`** in the **config cell** above, **or**
- In any cell: **`%env PHONE_IP=192.168.x.x`** (IPython magic), **or**
- Start Jupyter from a shell where you already **`export PHONE_IP=…`**.

**Avoid** **`! export PHONE_IP=…`** — that only touches a short-lived shell, not Python.


In [9]:
import os
import urllib.request

# Resolves same way as runner: PHONE_URL overrides host:8080
_ip = (os.environ.get("PHONE_IP") or "").strip()
_url = (os.environ.get("PHONE_URL") or "").strip().rstrip("/")
if _url:
    base = _url
elif _ip:
    base = f"http://{_ip}:8080"
else:
    raise RuntimeError(
        "PHONE_IP / PHONE_URL not set — set `_phone_ip` in the config cell (or `%env PHONE_IP=…`), "
        "not `!export …`."
    )

print(f"Checking {base}/health")
body = urllib.request.urlopen(f"{base}/health", timeout=10).read()
print(body.decode("utf-8", errors="replace"))


Checking http://192.168.86.241:8080/health
{"status":"ok","model":"gemma4-e2b","timestamp":"2026-05-11T22:17:16.361912"}


## Scenario — **`generic`**

**Condition A** — spike-style field names; scorer maps `key_date` → `response_deadline` for the critical hallucination metric. Source: [`research/eval/prompt_conditions.py`](../eval/prompt_conditions.py), `PROMPT_ABLATION_VERSION` **2026-04-11** (`build_prompt_generic`).

User message sent to **`/infer`** (OCR + notice content appended on device):

```
You are a document analysis assistant. Read the document carefully.

Extract information into the JSON object below. Use only information visible in the document.

Return ONLY valid JSON with exactly these keys. No markdown fences, no commentary.

{
  "document_type": "",
  "holder_name": "",
  "key_date": "",
  "secondary_date": "",
  "key_amount_or_address": "",
  "any_id_or_case_number": "",
}

For any field you cannot read clearly, set its value to UNREADABLE.

```

Runs **80** `/infer` calls (20 × 4 variants).

Output JSONL: **`results/ablation_generic_<STAMP>.jsonl`**.


In [10]:
run_scenario("generic")


/Users/danfinkel/github/civics/.venv/bin/python /Users/danfinkel/github/civics/research/eval/runner.py --artifacts D01 --track a --variants clean,clean_jpeg,degraded,blurry --prompt-condition generic --runs 20 --temp 0.0 --cooldown 2.0 --condition-break-s 0 --out /Users/danfinkel/github/civics/research/prompt_ablation/results/ablation_generic_20260412_notebook.jsonl
Health OK (round 1/3): /health mean 149.6ms over 3 pings (max 200ms)
  generic D01 clean run 20/20
  generic D01 clean_jpeg run 20/20
  generic D01 degraded run 20/20
  generic D01 blurry run 20/20
{
  "n_runs": 80,
  "n_scored": 80,
  "parse_ok_rate": 1.0,
  "avg_score_mean": -0.25,
  "avg_score_std": 0.4330127018922193,
  "hallucination_rate": 0.4583333333333333,
  "exact_rate": 0.20833333333333334,
  "partial_rate": 0.0,
  "format_mismatch_rate": 0.0,
  "transcription_error_rate": 0.041666666666666664,
  "semantic_paraphrase_rate": 0.0,
  "verbatim_quote_rate": 0.041666666666666664,
  "misattribution_rate": 0.20833333333

PosixPath('/Users/danfinkel/github/civics/research/prompt_ablation/results/ablation_generic_20260412_notebook.jsonl')

Sleep between scenarios reduces thermal throttling (`CONDITION_BREAK_S`). Executing the **next cell** waits, then launches **semantic**.


In [ ]:
condition_break()


## Scenario — **`semantic`**

**Condition B** — production-style semantic keys + date disambiguation; critical field **`response_deadline`**. Same source/version as generic (`build_prompt_semantic`).

User message sent to **`/infer`**:

```
You are a document analysis assistant for a Massachusetts civic benefits notice.

Rules:
- Extract only values clearly present in the document.
- Read each field from its labeled location in the notice.
- For date fields: use semantic labels as requested in the JSON keys (e.g. notice_date vs response_deadline). Do not swap due dates with notice dates.
- Copy names character-by-character as printed.
- If you cannot read a field, set its value to UNREADABLE.
- Return ONLY valid JSON with exactly these keys. No markdown.

{
  "notice_date": "",
  "response_deadline": "",
  "requested_category": "",
  "consequence": "",
  "caseworker_name": "",
  "case_number": "",
  "recipient_name": "",
}

```

Output: **`results/ablation_semantic_<STAMP>.jsonl`**.


In [11]:
run_scenario("semantic")


/Users/danfinkel/github/civics/.venv/bin/python /Users/danfinkel/github/civics/research/eval/runner.py --artifacts D01 --track a --variants clean,clean_jpeg,degraded,blurry --prompt-condition semantic --runs 20 --temp 0.0 --cooldown 2.0 --condition-break-s 0 --out /Users/danfinkel/github/civics/research/prompt_ablation/results/ablation_semantic_20260412_notebook.jsonl
Health OK (round 1/3): /health mean 83.8ms over 3 pings (max 200ms)
  semantic D01 clean run 20/20
  semantic D01 clean_jpeg run 20/20
  semantic D01 degraded run 20/20
  semantic D01 blurry run 20/20
{
  "n_runs": 80,
  "n_scored": 80,
  "parse_ok_rate": 1.0,
  "avg_score_mean": 0.571425,
  "avg_score_std": 1.0448815970601644,
  "hallucination_rate": 0.2857142857142857,
  "exact_rate": 0.39285714285714285,
  "partial_rate": 0.0,
  "format_mismatch_rate": 0.10714285714285714,
  "transcription_error_rate": 0.07142857142857142,
  "semantic_paraphrase_rate": 0.0,
  "verbatim_quote_rate": 0.07142857142857142,
  "misattributio

PosixPath('/Users/danfinkel/github/civics/research/prompt_ablation/results/ablation_semantic_20260412_notebook.jsonl')

## Cooldown → **`semantic-preview`**

Preview path is heavier; give the handset a pause unless `CONDITION_BREAK_S=0`.


In [ ]:
condition_break()


## Scenario — **`semantic-preview`**

**Condition C** — per [`prompt_conditions.py`](../eval/prompt_conditions.py): **same extraction user message as `semantic`** (`build_prompt_semantic` / `build_extraction_prompt("semantic-preview")`). The behavioral difference vs **semantic** is **on-device**: a notice-first preview path before the extraction call (design notes in `prompt_conditions.py` module docstring).

Output: **`results/ablation_semantic_preview_<STAMP>.jsonl`**.

For **generic vs semantic** plots below you only need the first two JSONLs; this scenario is optional for parity with [`run_prompt_ablation.sh`](./run_prompt_ablation.sh).


In [ ]:
run_scenario("semantic-preview")


---

## Figures for markdown export

Execute the next cell to refresh PNGs (**stable filenames**).

Then the markdown sections below render in Jupyter / VS Code (paths are relative to **`research/prompt_ablation/`**).

Metrics: fraction of rows where **`critical_label == hallucinated`** (D01 deadline field), grouped by **`variant`**.


In [ ]:
import sys

if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

from ablation_plots import (
    load_jsonl,
    plot_generic_vs_semantic_grouped,
    plot_single_condition_variant_bars,
)

for label, path in [
    ("generic", PATH_GENERIC),
    ("semantic", PATH_SEMANTIC),
]:
    if not path.exists():
        raise FileNotFoundError(path)

_gen = load_jsonl(PATH_GENERIC)
_sem = load_jsonl(PATH_SEMANTIC)

plot_single_condition_variant_bars(
    _gen,
    title="prompt_condition=generic — critical hallucinated rate by variant",
    save_path=FIG_DIR / "d01_halluc_by_variant_generic.png",
)
plot_single_condition_variant_bars(
    _sem,
    title="prompt_condition=semantic — critical hallucinated rate by variant",
    save_path=FIG_DIR / "d01_halluc_by_variant_semantic.png",
)
plot_generic_vs_semantic_grouped(
    _gen,
    _sem,
    save_path=FIG_DIR / "d01_halluc_generic_vs_semantic.png",
)

print("Wrote:")
for p in sorted(FIG_DIR.glob("*.png")):
    print(" ", p.relative_to(LAB_DIR))


### Generic (`prompt_condition=generic`)

**Prompt (Condition A — verbatim)** — same as **`build_prompt_generic()`** ([`prompt_conditions.py`](../eval/prompt_conditions.py), v **2026-04-11**):

```
You are a document analysis assistant. Read the document carefully.

Extract information into the JSON object below. Use only information visible in the document.

Return ONLY valid JSON with exactly these keys. No markdown fences, no commentary.

{
  "document_type": "",
  "holder_name": "",
  "key_date": "",
  "secondary_date": "",
  "key_amount_or_address": "",
  "any_id_or_case_number": "",
}

For any field you cannot read clearly, set its value to UNREADABLE.

```

**Metric:** fraction of Monte Carlo rows where **`critical_label`** is **`hallucinated`** (**`key_date`** vs ground truth deadline), per **`variant`**.

![](results/figures/d01_halluc_by_variant_generic.png)


### Semantic (`prompt_condition=semantic`)

**Prompt (Condition B — verbatim)** — **`build_prompt_semantic()`**:

```
You are a document analysis assistant for a Massachusetts civic benefits notice.

Rules:
- Extract only values clearly present in the document.
- Read each field from its labeled location in the notice.
- For date fields: use semantic labels as requested in the JSON keys (e.g. notice_date vs response_deadline). Do not swap due dates with notice dates.
- Copy names character-by-character as printed.
- If you cannot read a field, set its value to UNREADABLE.
- Return ONLY valid JSON with exactly these keys. No markdown.

{
  "notice_date": "",
  "response_deadline": "",
  "requested_category": "",
  "consequence": "",
  "caseworker_name": "",
  "case_number": "",
  "recipient_name": "",
}

```

**Metric:** same plot for **`critical_label`** with **`response_deadline`** grounded against GT.

![](results/figures/d01_halluc_by_variant_semantic.png)


### Generic vs semantic (grouped bars)

Side-by-side rates by **`variant`** for quick comparison across prompt styles.

![](results/figures/d01_halluc_generic_vs_semantic.png)
